In [1]:
# ══════════════════════════════════════════════════════════════════
# NOTEBOOK 3 : INDICATOR SCORING
# Follows  exact thresholds from reference model
# Converts Prophet deviation/ratio back to raw values first
#
# thresholds:
#   STU:    25 / 20 / 15 / 0      (raw %)
#   Demand: 10 / 12 / 14 / 16     (ratio-based, see below)
#   PPI:     3 /  0 / -3 / -6     (raw deviation)
#   KSA:    15 / 25 / 50          (raw %)
#   Policy: 10 / 40 / 70 / 100    (raw score)
#   BDI:    30 / 60 / 90          (0-100 normalised)
#
# Output tables:
#   srm.prophet_indicator_scores
#   srm.prophet_policy_scores
# ══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

COMMODITIES   = ["Wheat","Corn","Rice","Soybean","Barley"]
CURRENT_MONTH = pd.Timestamp("2026-07-01")
FORECAST_MONTHS = pd.to_datetime(
    ["2026-08-01","2026-09-01","2026-10-01"]
)

STRING_COLS = [
    "ksa_top1_country","ksa_top1_country_lag1",
    "ksa_top3_countries","top_buyer_country",
    "individually_tracked_countries"
]

# WEIGHTS = {
#     "stu_deviation":         0.18,   # was 0.20
#     "demand_ratio":          0.12,   # was 0.13
#     "ppi_deviation":         0.11,   # was 0.12
#     "ksa_deviation":         0.13,   # was 0.15
#     "policy_risk_weighted":  0.18,   # was 0.20
#     "bdi_ratio":             0.18,   # was 0.20
#     "domestic_reserves_gap": 0.10    # NEW
# }

# WEIGHTS = {
#     "stu_deviation":         0.15,
#     "demand_ratio":          0.10,
#     "ppi_deviation":         0.15,
#     "ksa_deviation":         0.10,
#     "policy_risk_weighted":  0.20,
#     "bdi_ratio":             0.20,
#     "domestic_reserves_gap": 0.10
# }

WEIGHTS = {
    "stu_deviation":         0.15,   # unchanged
    "demand_ratio":          0.05,   # was 0.10
    "ppi_deviation":         0.05,   # was 0.15  
    "ksa_deviation":         0.15,   # was 0.10
    "policy_risk_weighted":  0.15,   # was 0.20
    "bdi_ratio":             0.20,   # unchanged
    "domestic_reserves_gap": 0.15,   # was 0.10
    "fob_price_deviation":   0.10,   # NEW
}

# INDICATORS_ORDERED = [
#     "stu_deviation","demand_ratio","ppi_deviation",
#     "ksa_deviation","policy_risk_weighted","bdi_ratio",
#     "domestic_reserves_gap"   # NEW
# ]

INDICATORS_ORDERED = [
    "stu_deviation","demand_ratio","ppi_deviation",
    "ksa_deviation","policy_risk_weighted","bdi_ratio",
    "domestic_reserves_gap","fob_price_deviation"   # NEW
]

INDICATOR_LABELS = {
    "stu_deviation":         "Global Stock-to-Use Ratio",
    "demand_ratio":          "Import Demand Pressure",
    "ppi_deviation":         "Production Potential Index",
    "ksa_deviation":         "KSA Import Concentration",
    "policy_risk_weighted":  "Export Restriction Status",
    "bdi_ratio":             "Logistic Disruption Index",
    "domestic_reserves_gap": "Domestic Strategic Reserves",
    "fob_price_deviation":   "FOB Price"   # NEW
}

def save_to_lakehouse(df_pandas, table_name, schema="srm"):
    full_name = f"{schema}.{table_name}"
    # Clean column names
    df_clean = df_pandas.copy()
    df_clean.columns = [
        c.replace("%","pct").replace("+","p").replace("/","_").replace(" ","_")
        for c in df_clean.columns
    ]
    spark.createDataFrame(df_clean) \
         .write.mode("overwrite") \
         .option("overwriteSchema","true") \
         .format("delta") \
         .saveAsTable(full_name)
    count = spark.table(full_name).count()
    print(f"✓ {full_name}: {count} rows saved")

def load_table(table_name, drop_strings=False):
    df_spark = spark.table(f"srm.{table_name}")
    if drop_strings:
        drop_cols = [c for c in STRING_COLS if c in df_spark.columns]
        if drop_cols:
            df_spark = df_spark.drop(*drop_cols)
    df = df_spark.toPandas()
    for col in df.columns:
        if col in ["ds","year_month"]:
            df[col] = pd.to_datetime(df[col])
    return df

print("=== Notebook 3: Indicator Scoring ===")

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 3, Finished, Available, Finished, False)

=== Notebook 3: Indicator Scoring ===


In [2]:
# ══════════════════════════════════════════════════════════════════
# CHOKEPOINT EXPOSURE — dampens/amplifies BDI score per commodity
# based on which countries actually supply it and current chokepoint severity
# ══════════════════════════════════════════════════════════════════

# ── Country → chokepoint route exposure (0-1 scale) ──────────────
# Best-effort geographic reasoning
COUNTRY_CHOKEPOINT_EXPOSURE = {
    # Black Sea/Mediterranean — ONLY route to Jeddah is via Suez/Red Sea,
    # no alternative exists. High dependency on that one route.
    "Romania":    {"red_sea": 0.8, "bab_el_mandeb": 0.0, "hormuz": 0.0},
    "Russia":     {"red_sea": 0.8, "bab_el_mandeb": 0.0, "hormuz": 0.0},
    "France":     {"red_sea": 0.8, "bab_el_mandeb": 0.0, "hormuz": 0.0},

    # South America — Cape route to Jeddah MUST pass through Bab-el-Mandeb,
    # no alternative. Minimal Gulf-port trade.
    "Argentina":  {"red_sea": 0.0, "bab_el_mandeb": 0.85, "hormuz": 0.10},
    "Brazil":     {"red_sea": 0.0, "bab_el_mandeb": 0.85, "hormuz": 0.10},

    # North America — genuine routing OPTIONALITY (Suez or Cape), so real
    # dependency on any single chokepoint is lower than countries with no choice
    "USA":        {"red_sea": 0.4, "bab_el_mandeb": 0.3, "hormuz": 0.15},
    "Canada":     {"red_sea": 0.4, "bab_el_mandeb": 0.3, "hormuz": 0.15},

    # South Asia — serves BOTH Saudi coasts directly, no alternative for
    # either: Red Sea ports via Bab-el-Mandeb, Gulf ports via Hormuz
    "India":      {"red_sea": 0.0, "bab_el_mandeb": 0.85, "hormuz": 0.85},
    "Pakistan":   {"red_sea": 0.0, "bab_el_mandeb": 0.75, "hormuz": 0.90},

    "_DEFAULT":   {"red_sea": 0.4, "bab_el_mandeb": 0.4, "hormuz": 0.30},
}

# ── Manually maintained — update as chokepoint situations evolve ─
# 0.0 = no active disruption, higher = more severe
# Log changes with date + reason so history is auditable
CHOKEPOINT_SEVERITY = {
    "red_sea":       0.0,
    "bab_el_mandeb":  0.0,
    "hormuz":         0.0,
}


def calculate_commodity_exposure(top1_country, top3_countries, top1_share, top3_share, severity_weights):
    """Weighted chokepoint exposure score (0-1) for one commodity, based on its actual supplier mix."""
    if pd.isna(top1_share) or not top3_countries:
        return None

    remaining = max(top3_share - top1_share, 0)
    per_other = remaining / 2 if len(top3_countries) > 1 else 0
    shares = [top1_share] + [per_other] * (len(top3_countries) - 1)

    per_chokepoint = {"red_sea": 0, "bab_el_mandeb": 0, "hormuz": 0}
    for country, share in zip(top3_countries, shares):
        exposure = COUNTRY_CHOKEPOINT_EXPOSURE.get(country, COUNTRY_CHOKEPOINT_EXPOSURE["_DEFAULT"])
        for key in per_chokepoint:
            per_chokepoint[key] += exposure[key] * share

    total_weight = sum(severity_weights.values())
    if total_weight > 0:
        resultant_score = sum(
            per_chokepoint[k] * severity_weights.get(k, 0) for k in per_chokepoint
        ) / total_weight
    else:
        resultant_score = 0.0

    result = {k: round(v, 3) for k, v in per_chokepoint.items()}
    result["resultant_score"] = round(resultant_score, 3)
    return result

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 4, Finished, Available, Finished, False)

In [3]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: LOAD DATA
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 1: Loading data ===")

df_forecasts   = load_table("prophet_forecasts")
df_current     = load_table("prophet_current_values")
df_demand_curr = load_table("prophet_demand_current")
df_policy      = load_table("features_policy", drop_strings=True)
df_baseline    = load_table("prophet_baseline_summary")

print(f"Forecasts:      {df_forecasts.shape}")
print(f"Current values: {df_current.shape}")
print(f"Demand rolling: {df_demand_curr.shape}")
print(f"Policy:         {df_policy.shape}")
print(f"Baseline:       {df_baseline.shape}")

# ── Build baseline lookup dict ─────────────────────────────────────
# {(commodity, indicator): baseline_avg}
baseline_lookup = {}
for _, row in df_baseline.iterrows():
    key = (row["commodity"], row["indicator"])
    baseline_lookup[key] = float(row["baseline_avg"])

print(f"\nBaseline lookup entries: {len(baseline_lookup)}")
print("Sample baselines:")
for (commodity, indicator), avg in list(baseline_lookup.items())[:6]:
    print(f"  {commodity:<8} {indicator:<20} baseline = {avg:.3f}")

# ── Add demand into current and forecast ──────────────────────────
df_demand_for_current = df_demand_curr[
    df_demand_curr["period_type"]=="current"
][["ds","yhat","yhat_lower","yhat_upper","commodity"]].copy()
df_demand_for_current["indicator"]   = "demand_ratio"
df_demand_for_current["period_type"] = "current"
df_demand_for_current["ds"] = df_demand_for_current["ds"].astype("datetime64[us]")
df_current["ds"] = df_current["ds"].astype("datetime64[us]")

df_current = pd.concat(
    [df_current, df_demand_for_current], ignore_index=True
).drop_duplicates(
    subset=["ds","indicator","commodity"], keep="last"
).reset_index(drop=True)

df_demand_for_forecast = df_demand_curr[
    df_demand_curr["period_type"]=="forecast"
][["ds","yhat","yhat_lower","yhat_upper","commodity"]].copy()
df_demand_for_forecast["indicator"] = "demand_ratio"
df_demand_for_forecast["ds"] = df_demand_for_forecast["ds"].astype("datetime64[us]")
df_forecasts["ds"] = df_forecasts["ds"].astype("datetime64[us]")

df_forecasts = pd.concat(
    [df_forecasts, df_demand_for_forecast], ignore_index=True
).reset_index(drop=True)

# ── Combine current + forecast ─────────────────────────────────────
df_current["period_type"]   = "current"
df_forecasts["period_type"] = "forecast"

df_to_score = pd.concat([
    df_current[["ds","yhat","yhat_lower","yhat_upper",
                "indicator","commodity","period_type"]],
    df_forecasts[["ds","yhat","yhat_lower","yhat_upper",
                  "indicator","commodity","period_type"]]
], ignore_index=True)

df_to_score["ds"] = pd.to_datetime(df_to_score["ds"])
df_to_score = df_to_score.sort_values(
    ["commodity","indicator","ds"]
).reset_index(drop=True)

print(f"\nCombined scoring table: {df_to_score.shape}")
print(f"Indicators: {sorted(df_to_score['indicator'].unique())}")

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 5, Finished, Available, Finished, False)


=== Step 1: Loading data ===
Forecasts:      (75, 7)
Current values: (35, 9)
Demand rolling: (20, 10)
Policy:         (360, 21)
Baseline:       (30, 6)

Baseline lookup entries: 30
Sample baselines:
  Barley   demand_ratio         baseline = 3606199.490
  Barley   ppi_deviation        baseline = 99.421
  Barley   ksa_deviation        baseline = 81.994
  Barley   fob_price_deviation  baseline = 238.517
  Wheat    ppi_deviation        baseline = 100.620
  Wheat    ksa_deviation        baseline = 81.856

Combined scoring table: (125, 7)
Indicators: ['bdi_ratio', 'demand_ratio', 'fob_price_deviation', 'ksa_deviation', 'policy_risk_weighted', 'ppi_deviation', 'stu_deviation']


In [4]:
# ── Step 1b: Load domestic reserves table ─────────────────────────
print("\n=== Step 1b: Loading domestic reserves ===")

df_reserves_raw = spark.table("srm.gld_domestic_reserves").toPandas()
df_reserves_raw.columns = [c.strip() for c in df_reserves_raw.columns]

# Rename columns for clarity
df_reserves_raw = df_reserves_raw.rename(columns={
    "Commodity":                      "commodity",
    "Target_Strategic_Reserve_KT_In_Months": "target_months",
    "Current_Stock_In_Months":        "current_months",
    "Reported_Date":                  "reported_date"
})

df_reserves_raw["reported_date"] = pd.to_datetime(df_reserves_raw["reported_date"])

print(f"Reserves table: {df_reserves_raw.shape}")
print(df_reserves_raw.to_string(index=False))

# Compute gap per commodity
df_reserves_raw["gap_months"] = (
    df_reserves_raw["target_months"] - df_reserves_raw["current_months"]
)

print(f"\nGap analysis (target - current, positive = below target = risk):")
print(df_reserves_raw[["commodity","target_months","current_months","gap_months"]].to_string(index=False))

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 6, Finished, Available, Finished, False)


=== Step 1b: Loading domestic reserves ===
Reserves table: (5, 4)
commodity  target_months  current_months reported_date
  Soybean            3.0             3.2    2026-06-24
   Barley            3.0             3.5    2026-06-24
    Wheat            4.0             4.0    2026-06-24
     Corn            3.0             3.4    2026-06-24
     Rice            7.0             8.8    2026-06-24

Gap analysis (target - current, positive = below target = risk):
commodity  target_months  current_months  gap_months
  Soybean            3.0             3.2        -0.2
   Barley            3.0             3.5        -0.5
    Wheat            4.0             4.0         0.0
     Corn            3.0             3.4        -0.4
     Rice            7.0             8.8        -1.8


In [5]:
# ══════════════════════════════════════════════════════════════════
# STEP 1c: BUILD CHOKEPOINT EXPOSURE MULTIPLIER PER COMMODITY
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 1c: Chokepoint exposure per commodity ===")

df_master_ksa = spark.table("srm.features_master").toPandas()
df_master_ksa["year_month"] = pd.to_datetime(df_master_ksa["year_month"])

# Most recent month where real (non-forward-filled) country data exists
dc_ksa_real = df_master_ksa[df_master_ksa["ksa_top1_country"].notna()] \
    .sort_values("year_month").groupby("commodity").tail(1)

COMMODITY_CHOKEPOINT_MULTIPLIER = {}

print(f"{'Commodity':<10} {'Red Sea':>10} {'Bab-el-Mandeb':>15} {'Hormuz':>10} {'RESULTANT':>12}")
print("-" * 62)
for _, row in dc_ksa_real.iterrows():
    countries = eval(row["ksa_top3_countries"]) if isinstance(row["ksa_top3_countries"], str) else row["ksa_top3_countries"]
    exposure = calculate_commodity_exposure(
        row["ksa_top1_country"], countries, row["ksa_top1_share"], row["ksa_top3_share"], CHOKEPOINT_SEVERITY
    )
    if exposure:
        COMMODITY_CHOKEPOINT_MULTIPLIER[row["commodity"]] = exposure["resultant_score"]
        print(f"{row['commodity']:<10} {exposure['red_sea']:>10.3f} {exposure['bab_el_mandeb']:>15.3f} {exposure['hormuz']:>10.3f} {exposure['resultant_score']:>12.3f}")

# Fallback for any commodity missing exposure data — assume full exposure (conservative, doesn't understate risk)
for c in COMMODITIES:
    if c not in COMMODITY_CHOKEPOINT_MULTIPLIER:
        COMMODITY_CHOKEPOINT_MULTIPLIER[c] = 1.0
        print(f"⚠ {c}: no exposure data found, defaulting multiplier to 1.0 (full exposure, conservative)")

# ── Save exposure multiplier to OneLake so downstream notebooks
df_chokepoint_exposure = pd.DataFrame([
    {"commodity": c, "chokepoint_multiplier": m}
    for c, m in COMMODITY_CHOKEPOINT_MULTIPLIER.items()
])
save_to_lakehouse(df_chokepoint_exposure, "prophet_chokepoint_exposure")
print(f"\n✓ srm.prophet_chokepoint_exposure: {len(df_chokepoint_exposure)} rows saved")

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 7, Finished, Available, Finished, False)


=== Step 1c: Chokepoint exposure per commodity ===
Commodity     Red Sea   Bab-el-Mandeb     Hormuz    RESULTANT
--------------------------------------------------------------
Rice            0.032           0.739      0.739        0.000
Barley          0.240           0.222      0.026        0.000
Corn            0.095           0.697      0.109        0.000
Soybean         0.325           0.403      0.141        0.000
Wheat           0.513           0.195      0.023        0.000
✓ srm.prophet_chokepoint_exposure: 5 rows saved

✓ srm.prophet_chokepoint_exposure: 5 rows saved


In [6]:
# ── Save severity to OneLake so Notebook 5 can check for full closure ─
df_chokepoint_severity = pd.DataFrame([
    {"chokepoint": k, "severity": v} for k, v in CHOKEPOINT_SEVERITY.items()
])
save_to_lakehouse(df_chokepoint_severity, "prophet_chokepoint_severity")

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 8, Finished, Available, Finished, False)

✓ srm.prophet_chokepoint_severity: 3 rows saved


In [7]:
BDI_WEIGHT_MULTIPLIER_ON_FULL_CLOSURE = 2.0
FULL_CLOSURE_THRESHOLD = 1.0

def get_effective_weights(base_weights, full_closure):
    """If Bab-el-Mandeb AND Hormuz are both at severity 1.0 (fully closed),
    double BDI's weight. Total weight is allowed to exceed 100% —
    deliberate, so a full closure can push the composite score higher
    than the normal weighting scheme permits. No other weight is reduced."""
    if not full_closure:
        return base_weights
    weights = dict(base_weights)
    weights["bdi_ratio"] = weights["bdi_ratio"] * BDI_WEIGHT_MULTIPLIER_ON_FULL_CLOSURE
    return weights

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 9, Finished, Available, Finished, False)

In [8]:
FULL_CLOSURE = (
    CHOKEPOINT_SEVERITY.get("bab_el_mandeb", 0) >= FULL_CLOSURE_THRESHOLD and
    CHOKEPOINT_SEVERITY.get("hormuz", 0) >= FULL_CLOSURE_THRESHOLD
)
EFFECTIVE_WEIGHTS = get_effective_weights(WEIGHTS, FULL_CLOSURE)

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 10, Finished, Available, Finished, False)

In [9]:
# ══════════════════════════════════════════════════════════════════
# STEP 2: DEFINE SCORING FUNCTION 
# Converts Prophet output back to raw value first
# Then applies exact thresholds from image
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 2: Defining scoring function ===")

def interpolate(val, lo_thresh, hi_thresh, lo_score, hi_score):
    """Linear interpolation within a band."""
    if hi_thresh == lo_thresh:
        return float(lo_score)
    t = (val - lo_thresh) / (hi_thresh - lo_thresh)
    t = max(0.0, min(1.0, t))
    return round(lo_score + t * (hi_score - lo_score), 2)


def score_indicator(prophet_value, indicator, commodity,
                    baseline_lookup=baseline_lookup):
    """
    framework scoring.
    Step 1: Convert Prophet deviation/ratio → raw value
    Step 2: Apply thresholds from image

    Boss thresholds (Low/Watch/Warning/Emergency), UPDATED:
      STU:    -2.5 / -5.0 / -7.5 / -10.0   raw % deviation, interpolated
      Demand:  0 / 10 / 25                  raw % deviation, interpolated
      PPI:     3 / 0 / -3                   raw deviation, FLAT/STEPPED (30/55/75/100)
      KSA:    15 / 25 / 50                  raw %, interpolated
      Policy: 10 / 40 / 70                  raw score 0-100 — UNCHANGED
      BDI:    30 / 60 / 90                  normalised 0-100 score — UNCHANGED
      Reserves: 100 / 75 / 50               % of target — handled separately in score_reserves_pct()
      FOB Price: 5 / 10 / 20                raw % deviation, interpolated — NEW
    """
    if pd.isna(prophet_value):
        return np.nan

    # ── STEP 1: Convert Prophet output to raw value ────────────────

    if indicator == "stu_deviation":
        baseline = baseline_lookup.get((commodity, "stu_ratio"), 26.86)

        # % deviation from 5Y baseline (signed)
        # positive = stocks ABOVE average = good = low score
        # negative = stocks BELOW average = risk = high score
        pct_dev = (prophet_value / baseline) * 100

        # ── UPDATED thresholds (Venkat) ──
        # Low:        pct_dev >= -2.5%
        # Watch:      -2.5% to -5.0%
        # Warning:    -5.0% to -7.5%
        # Emergency:  -7.5% to -10.0% (capped beyond -10%)
        STU_DEV_FLOOR_PCT = -10.0   # Emergency threshold now doubles as the scoring floor

        if pct_dev >= -2.5:
            return 0.0
        elif pct_dev >= -5.0:
            return interpolate(pct_dev, -2.5, -5.0, 0, 40)
        elif pct_dev >= -7.5:
            return interpolate(pct_dev, -5.0, -7.5, 40, 70)
        else:
            return interpolate(max(pct_dev, STU_DEV_FLOOR_PCT), -7.5, STU_DEV_FLOOR_PCT, 70, 100)

    elif indicator == "ppi_deviation":
        # Prophet gave: deviation from baseline (ppi_base - baseline_avg)
        # ppi_base is already centred around 100
        baseline = baseline_lookup.get((commodity, "ppi_deviation"), 100.0)
        raw = prophet_value + baseline - 100  # convert to deviation from 100

        # ── UPDATED thresholds (Venkat, EWS slide) — flat score per band, no interpolation ──
        # Normal:   raw >= 3%           -> 30
        # Watch:    0% <= raw < 3%      -> 55
        # Warning: -3% < raw < 0%       -> 75
        # Emergency: raw <= -3%         -> 100  (floor — note strict "> -3" on Warning
        #             so -3% itself lands in Emergency, matching "<= -3%" on the slide)
        if raw >= 3:
            return 30.0
        elif raw >= 0:
            return 55.0
        elif raw > -3:
            return 75.0
        else:
            return 100.0

    elif indicator == "bdi_ratio":
        # ── UNCHANGED — left exactly as-is per instruction ──
        raw = min(100, max(0, (prophet_value - 0.5) / 1.5 * 100))

        BDI_NEUTRAL_SCORE = 33.33
        multiplier = COMMODITY_CHOKEPOINT_MULTIPLIER.get(commodity, 1.0)

        BDI_AMPLIFICATION_FACTOR = 2.0
        amplified_raw = BDI_NEUTRAL_SCORE + (raw - BDI_NEUTRAL_SCORE) * BDI_AMPLIFICATION_FACTOR
        amplified_raw = min(amplified_raw, 100)
        raw = BDI_NEUTRAL_SCORE + (amplified_raw - BDI_NEUTRAL_SCORE) * multiplier

        if raw < 30:    return 0.0
        elif raw < 60:  return interpolate(raw, 30, 60, 0, 50)
        elif raw < 90:  return interpolate(raw, 60, 90, 50, 90)
        else:           return interpolate(min(raw, 100), 90, 100, 90, 100)

    elif indicator == "demand_ratio":
        pct = (prophet_value - 1.0) * 100   # ratio -> % deviation from baseline

        # ── UPDATED thresholds (Venkat) — interpolated between EWS slide's band edges ──
        DEMAND_FLOOR_PCT = -25   # ⚠ assumption — % below which score flat-lines at floor; adjust if needed

        if pct < 0:
            return interpolate(max(pct, DEMAND_FLOOR_PCT), DEMAND_FLOOR_PCT, 0, 10, 30)
        elif pct < 10:
            return interpolate(pct, 0, 10, 30, 55)
        elif pct < 25:
            return interpolate(pct, 10, 25, 55, 75)
        else:
            return 100.0

    elif indicator == "ksa_deviation":
        # Prophet gave: deviation = current% - baseline%
        baseline = baseline_lookup.get((commodity, "ksa_top3_pct"),
                    baseline_lookup.get((commodity, "ksa_deviation"), 85.0))
        raw = prophet_value + baseline

        # ── UPDATED thresholds (Venkat, EWS slide) — interpolated between band edges ──
        if raw < 15:
            return interpolate(raw, 0, 15, 0, 30)
        elif raw < 25:
            return interpolate(raw, 15, 25, 30, 55)
        elif raw < 50:
            return interpolate(raw, 25, 50, 55, 75)
        else:
            return interpolate(min(raw, 100), 50, 100, 75, 100)

    # elif indicator == "fob_price_deviation":
    #     # NEW — Prophet gave: y = current_price - baseline_avg (raw currency deviation)
    #     baseline = baseline_lookup.get((commodity, "fob_price_deviation"), 300.0)
    #     pct_dev = (prophet_value / baseline) * 100

    #     # ── NEW thresholds (Venkat, EWS slide) — interpolated between band edges ──
    #     # Higher price = worse (costlier imports = risk)
    #     PRICE_DEV_FLOOR_PCT = -25   # ⚠ assumption — most negative deviation% expected; adjust if needed

    #     if pct_dev <= 5:
    #         return interpolate(max(pct_dev, PRICE_DEV_FLOOR_PCT), PRICE_DEV_FLOOR_PCT, 5, 0, 30)
    #     elif pct_dev <= 10:
    #         return interpolate(pct_dev, 5, 10, 30, 55)
    #     elif pct_dev <= 20:
    #         return interpolate(pct_dev, 10, 20, 55, 75)
    #     else:
    #         return interpolate(min(pct_dev, 50), 20, 50, 75, 100)   # ⚠ assumption — 50% treated as deep-Emergency ceiling

    elif indicator == "fob_price_deviation":
        baseline = baseline_lookup.get((commodity, "fob_price_deviation"))
        if baseline is None: return np.nan
        price = np.exp(prophet_value)                     # NEW — reconstruct actual price from log-scale output
        pct_dev = ((price - baseline) / baseline) * 100    # CHANGED — deviation computed from reconstructed price

        PRICE_DEV_FLOOR_PCT = -25
        if pct_dev <= 5:
            return interpolate(max(pct_dev, PRICE_DEV_FLOOR_PCT), PRICE_DEV_FLOOR_PCT, 5, 0, 30)
        elif pct_dev <= 10:
            return interpolate(pct_dev, 5, 10, 30, 55)
        elif pct_dev <= 20:
            return interpolate(pct_dev, 10, 20, 55, 75)
        else:
            return interpolate(min(pct_dev, 50), 20, 50, 75, 100)

    elif indicator == "policy_risk_weighted":
        # ── UNCHANGED — left exactly as-is per instruction ──
        raw = prophet_value

        if raw < 10:    return 0.0
        elif raw < 40:  return interpolate(raw, 10, 40, 0, 30)
        elif raw < 70:  return interpolate(raw, 40, 70, 30, 60)
        else:           return interpolate(min(raw, 100), 70, 100, 60, 100)

    else:
        return np.nan


# ── Test scoring function ──────────────────────────────────────────
print("\nScoring verification :")
print(f"\n{'Indicator':<25} {'Prophet':>10} {'Raw':>10} {'Score':>8}  Expected")
print("-" * 72)

# UPDATED — full test block, rewritten for all new thresholds
tests = [
    # STU: pct_dev = (prophet_value / baseline) * 100, baseline(Wheat)=26.86
    ("stu_deviation",  "Wheat",  -0.6715, "pct_dev= -2.5% → expect   0.0  (Low threshold)"),
    ("stu_deviation",  "Wheat",  -1.343,  "pct_dev= -5.0% → expect  40.0  (Watch/Warning boundary)"),
    ("stu_deviation",  "Wheat",  -2.0145, "pct_dev= -7.5% → expect  70.0  (Warning/Emergency boundary)"),
    ("stu_deviation",  "Wheat",  -2.686,  "pct_dev=-10.0% → expect 100.0  (Emergency floor)"),

    # PPI: raw = deviation + baseline - 100, baseline=100 → prophet_value ≈ raw
    ("ppi_deviation",  "Wheat",   3.0,   "raw= +3   → expect  30.0  (Normal)"),
    ("ppi_deviation",  "Wheat",   0.0,   "raw=  0   → expect  55.0  (Watch)"),
    ("ppi_deviation",  "Wheat",  -2.9,   "raw= -2.9 → expect  75.0  (Warning, just above floor)"),
    ("ppi_deviation",  "Wheat",  -3.0,   "raw= -3   → expect 100.0  (Emergency floor — note: -3 itself is Emergency)"),

    # BDI — unchanged, same tests as before
    ("bdi_ratio",      "Wheat",   0.95,  "score=30 → expect  0.0  (at Low threshold)"),
    ("bdi_ratio",      "Wheat",   1.40,  "score=60 → expect 30.0  (at Watch threshold)"),
    ("bdi_ratio",      "Wheat",   1.85,  "score=90 → expect 60.0  (at Warning threshold)"),

    # KSA: raw = deviation + baseline(83.9 for Wheat)
    ("ksa_deviation",  "Wheat",  13.37,  "raw=97.3% → expect 98.65 (deep Emergency)"),
    ("ksa_deviation",  "Corn",   -0.07,  "raw=96.5% → expect 98.25 (deep Emergency)"),

    # Policy — unchanged, same tests as before
    ("policy_risk_weighted","Wheat",17.5, "raw=17.5 → expect  7.5  (Watch band, 10-40)"),
    ("policy_risk_weighted","Wheat",45.0, "raw=45.0 → expect 15.5  (Warning band, 40-70)"),

    # Demand: pct = (ratio - 1.0) * 100, floor -25%
    ("demand_ratio",   "Wheat",   0.75,   "pct=-25.0% → expect  10.0  (floor)"),
    ("demand_ratio",   "Wheat",   0.857,  "pct=-14.3% → expect  18.56 (below baseline)"),
    ("demand_ratio",   "Wheat",   1.0,    "pct=  0.0% → expect  30.0  (Normal/Watch boundary)"),
    ("demand_ratio",   "Wheat",   1.10,   "pct= 10.0% → expect  55.0  (Watch/Warning boundary)"),
    ("demand_ratio",   "Rice",    1.183,  "pct= 18.3% → expect  66.07 (Warning band)"),
    ("demand_ratio",   "Wheat",   1.25,   "pct= 25.0% → expect 100.0  (Emergency)"),

    # FOB Price (NEW): pct_dev = (prophet_value / baseline) * 100, baseline(Wheat)=270.77
    ("fob_price_deviation", "Wheat", -44.126, "pct_dev=-16.3% → expect   8.7  (current Wheat deviation, sanity check)"),
    ("fob_price_deviation", "Wheat",  13.5385,"pct_dev=  5.0% → expect  30.0  (Normal/Watch boundary)"),
    ("fob_price_deviation", "Wheat",  27.077, "pct_dev= 10.0% → expect  55.0  (Watch/Warning boundary)"),
    ("fob_price_deviation", "Wheat",  54.154, "pct_dev= 20.0% → expect  75.0  (Warning/Emergency boundary)"),
    ("fob_price_deviation", "Wheat",  94.7695,"pct_dev= 35.0% → expect  87.5  (deep Emergency)"),
]

for indicator, commodity, prophet_val, expected in tests:
    score = score_indicator(prophet_val, indicator, commodity)

    # Compute raw for display
    if indicator == "stu_deviation":
        b = baseline_lookup.get((commodity,"stu_ratio"),26.86)
        raw_disp = (prophet_val / b) * 100
    elif indicator == "ppi_deviation":
        b = baseline_lookup.get((commodity,"ppi_deviation"),100.0)
        raw_disp = prophet_val + b - 100
    elif indicator == "bdi_ratio":
        raw_disp = min(100, max(0,(prophet_val-0.5)/1.5*100))
    elif indicator == "ksa_deviation":
        b = baseline_lookup.get((commodity,"ksa_top3_pct"),
            baseline_lookup.get((commodity,"ksa_deviation"),85.0))
        raw_disp = prophet_val + b
        raw_disp = min(raw_disp, 100.0)
    elif indicator == "demand_ratio":
        raw_disp = (prophet_val - 1.0) * 100
    elif indicator == "fob_price_deviation":
        b = baseline_lookup.get((commodity,"fob_price_deviation"),300.0)
        raw_disp = (prophet_val / b) * 100
    else:
        raw_disp = prophet_val

    print(f"{indicator:<25} {prophet_val:>10.3f} {raw_disp:>10.3f} {score:>8.2f}  {expected}")

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 11, Finished, Available, Finished, False)


=== Step 2: Defining scoring function ===

Scoring verification :

Indicator                    Prophet        Raw    Score  Expected
------------------------------------------------------------------------
stu_deviation                 -0.671     -2.500     0.00  pct_dev= -2.5% → expect   0.0  (Low threshold)
stu_deviation                 -1.343     -5.000    39.99  pct_dev= -5.0% → expect  40.0  (Watch/Warning boundary)
stu_deviation                 -2.014     -7.500    69.99  pct_dev= -7.5% → expect  70.0  (Warning/Emergency boundary)
stu_deviation                 -2.686     -9.999    99.99  pct_dev=-10.0% → expect 100.0  (Emergency floor)
ppi_deviation                  3.000      3.620    30.00  raw= +3   → expect  30.0  (Normal)
ppi_deviation                  0.000      0.620    55.00  raw=  0   → expect  55.0  (Watch)
ppi_deviation                 -2.900     -2.280    75.00  raw= -2.9 → expect  75.0  (Warning, just above floor)
ppi_deviation                 -3.000     -2.380    

In [10]:
# ══════════════════════════════════════════════════════════════════
# STEP 3: SCORE ALL INDICATORS
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 3: Scoring all indicators ===")

df_to_score["indicator_score"] = df_to_score.apply(
    lambda r: score_indicator(r["yhat"], r["indicator"], r["commodity"]), axis=1
)
df_to_score["indicator_score_lower"] = df_to_score.apply(
    lambda r: score_indicator(r["yhat_lower"], r["indicator"], r["commodity"]), axis=1
)
df_to_score["indicator_score_upper"] = df_to_score.apply(
    lambda r: score_indicator(r["yhat_upper"], r["indicator"], r["commodity"]), axis=1
)

print(f"Scored rows: {len(df_to_score)}")
print(f"NULL scores: {df_to_score['indicator_score'].isnull().sum()}")

print(f"\nScore distribution per indicator:")
print(df_to_score.groupby("indicator")["indicator_score"].describe().round(2))

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 12, Finished, Available, Finished, False)


=== Step 3: Scoring all indicators ===
Scored rows: 125
NULL scores: 0

Score distribution per indicator:
                      count   mean    std    min    25%    50%    75%     max
indicator                                                                    
bdi_ratio              20.0   5.55   0.00   5.55   5.55   5.55   5.55    5.55
demand_ratio           20.0  34.29  23.06  12.79  17.11  18.58  56.89   66.09
fob_price_deviation    20.0  15.92  16.60   0.00   3.95  12.58  15.81   56.40
ksa_deviation          20.0  95.48   5.71  78.06  95.45  97.34  99.31  100.00
policy_risk_weighted    5.0   1.67   3.28   0.00   0.00   0.00   0.84    7.51
ppi_deviation          20.0  50.75  13.70  30.00  48.75  55.00  55.00   75.00
stu_deviation          20.0  20.00  41.04   0.00   0.00   0.00   0.00  100.00


In [11]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: POLICY SCORING 
# Current: actual policy_risk_score_weighted
# Forecast: 2% monthly decay
# thresholds: 10 / 40 / 70 / 100
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 4: Policy scoring ===")

POLICY_DECAY = 0.98
df_policy["year_month"] = pd.to_datetime(df_policy["year_month"])
policy_rows = []

for commodity in COMMODITIES:
    dc = df_policy[df_policy["commodity"]==commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)

    current_row = dc[dc["year_month"] <= CURRENT_MONTH].tail(1)
    if current_row.empty:
        continue

    current_val   = float(current_row["policy_risk_score_weighted"].values[0])
    current_score = score_indicator(current_val, "policy_risk_weighted", commodity)

    policy_rows.append({
        "ds":              CURRENT_MONTH,
        "yhat":            current_val,
        "indicator_score": current_score,
        "indicator_score_lower": current_score,
        "indicator_score_upper": current_score,
        "indicator":       "policy_risk_weighted",
        "commodity":       commodity,
        "period_type":     "current"
    })

    print(f"\n{commodity}: policy={current_val:.2f} → score={current_score:.2f}")

    for i, month in enumerate(FORECAST_MONTHS):
        decay_factor   = POLICY_DECAY ** (i + 1)
        forecast_val   = round(current_val * decay_factor, 4)
        forecast_score = score_indicator(
            forecast_val, "policy_risk_weighted", commodity
        )
        policy_rows.append({
            "ds":              month,
            "yhat":            forecast_val,
            "indicator_score": forecast_score,
            "indicator_score_lower": forecast_score,
            "indicator_score_upper": forecast_score,
            "indicator":       "policy_risk_weighted",
            "commodity":       commodity,
            "period_type":     "forecast"
        })
        print(f"  {month.strftime('%b %Y')}: val={forecast_val:.2f} "
              f"score={forecast_score:.2f}")

df_policy_scores = pd.DataFrame(policy_rows)
df_policy_scores["ds"] = pd.to_datetime(df_policy_scores["ds"])

save_to_lakehouse(df_policy_scores, "prophet_policy_scores")

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 13, Finished, Available, Finished, False)


=== Step 4: Policy scoring ===

Wheat: policy=17.51 → score=7.51
  Aug 2026: val=17.16 score=7.16
  Sep 2026: val=16.81 score=6.81
  Oct 2026: val=16.48 score=6.48

Corn: policy=4.95 → score=0.00
  Aug 2026: val=4.85 score=0.00
  Sep 2026: val=4.75 score=0.00
  Oct 2026: val=4.66 score=0.00

Rice: policy=10.84 → score=0.84
  Aug 2026: val=10.63 score=0.63
  Sep 2026: val=10.41 score=0.41
  Oct 2026: val=10.21 score=0.21

Soybean: policy=5.83 → score=0.00
  Aug 2026: val=5.71 score=0.00
  Sep 2026: val=5.60 score=0.00
  Oct 2026: val=5.49 score=0.00

Barley: policy=0.00 → score=0.00
  Aug 2026: val=0.00 score=0.00
  Sep 2026: val=0.00 score=0.00
  Oct 2026: val=0.00 score=0.00
✓ srm.prophet_policy_scores: 20 rows saved


In [12]:
# ══════════════════════════════════════════════════════════════════
# STEP 4b: DOMESTIC RESERVES SCORING
# Rule-based — no Prophet, no baseline
# Reserves deplete over the forecast horizon based on each commodity's
# chokepoint exposure — the fraction of normal import supply that fails
# to arrive each month must be covered from existing reserves.
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 4b: Domestic reserves scoring ===")

# UPDATED — thresholds per EWS slide — interpolated, direction reversed vs. most indicators
def score_reserves_pct(pct_of_target):
    """
    pct_of_target = (current_months / target_months) * 100
      >= 100%     -> Normal  (score ramps 0-30 as pct climbs from 100% to ceiling)
      75% - 100%  -> Watch   (score 30-55)
      50% - 75%   -> Warning (score 55-75)
      < 50%       -> Emergency (score 75-100, floors at 0% coverage)
    """
    if pd.isna(pct_of_target): return np.nan

    # ── UPDATED thresholds (Venkat, EWS slide) ──
    RESERVES_CEILING_PCT = 150   # ⚠ assumption — pct_of_target at/above which score bottoms out at 0

    if pct_of_target >= 100:
        return round(interpolate(min(pct_of_target, RESERVES_CEILING_PCT), RESERVES_CEILING_PCT, 100, 0, 30), 2)
    elif pct_of_target >= 75:
        return round(interpolate(pct_of_target, 100, 75, 30, 55), 2)
    elif pct_of_target >= 50:
        return round(interpolate(pct_of_target, 75, 50, 55, 75), 2)
    else:
        return round(interpolate(max(pct_of_target, 0), 50, 0, 75, 100), 2)

# Verify scoring function
# UPDATED — test cases match new 100/75/50 thresholds
print("\nReserves scoring verification:")
test_pcts = [(150.0,"ceiling → best score"), (100.0,"Normal→Watch boundary"),
             (87.5,"mid Watch"), (75.0,"Watch→Warning boundary"),
             (62.5,"mid Warning"), (50.0,"Warning→Emergency boundary"),
             (25.0,"mid Emergency"), (0.0,"no coverage, floor")]
for pct, label in test_pcts:
    print(f"  pct_of_target={pct:>6.2f}% → score={score_reserves_pct(pct):>6.2f}  ({label})")

# Score all periods
reserves_rows = []

for commodity in COMMODITIES:
    res = df_reserves_raw[df_reserves_raw["commodity"]==commodity]
    if res.empty:
        print(f"  WARNING: No reserves data for {commodity}")
        continue

    gap            = float(res["gap_months"].values[0])
    target         = float(res["target_months"].values[0])
    current        = float(res["current_months"].values[0])
    pct_of_target_current = round(current / target * 100, 2) if target else np.nan

    # ── Dynamic reserve depletion under disrupted supply ───────────
    # Each forecast month, the fraction of normal import supply that's
    # disrupted (this commodity's chokepoint exposure) fails to arrive.
    # That undelivered supply must be covered from existing reserves —
    # so reserves lose (exposure × 1 month of cover) for every month
    # the disruption continues. E.g. if a commodity is 90% exposed to
    # two fully-closed straits, ~90% of a month's normal supply is
    # missing, so ~0.9 months of reserve cover gets consumed that month.
    exposure = COMMODITY_CHOKEPOINT_MULTIPLIER.get(commodity, 1.0)

    print(f"\n{commodity}:")
    print(f"  Target: {target:.2f} months | Current: {current:.2f} months")
    print(f"  Gap:    {gap:.2f} months | Exposure (disrupted supply fraction): {exposure:.3f}")

    all_periods = [CURRENT_MONTH] + list(FORECAST_MONTHS)
    period_types = ["current","forecast","forecast","forecast"]

    for i, (period, ptype) in enumerate(zip(all_periods, period_types)):
        depleted_months = max(current - exposure * i, 0.0)
        pct_of_target = round(depleted_months / target * 100, 2) if target else np.nan
        score = score_reserves_pct(pct_of_target)
        print(f"  {ptype:>8} (t={i}): reserve_months={depleted_months:.2f}  pct_of_target={pct_of_target:.1f}%  score={score:.1f}")

        reserves_rows.append({
            "ds":              period,
            "indicator":       "domestic_reserves_gap",
            "commodity":       commodity,
            "period_type":     ptype,
            "yhat":            pct_of_target,
            "gap_months":      gap,
            "target_months":   target,
            "current_months":  depleted_months,
            "indicator_score": score,
            "indicator_score_lower": score,
            "indicator_score_upper": score,
            "note": f"Depleted {exposure*i:.2f} months of cover, based on chokepoint exposure {exposure:.3f}"
        })

df_reserves_scores = pd.DataFrame(reserves_rows)
df_reserves_scores["ds"] = pd.to_datetime(df_reserves_scores["ds"])

print(f"\nReserves scores: {df_reserves_scores.shape}")
save_to_lakehouse(df_reserves_scores, "prophet_reserves_scores")

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 14, Finished, Available, Finished, False)


=== Step 4b: Domestic reserves scoring ===

Reserves scoring verification:
  pct_of_target=150.00% → score=  0.00  (ceiling → best score)
  pct_of_target=100.00% → score= 30.00  (Normal→Watch boundary)
  pct_of_target= 87.50% → score= 42.50  (mid Watch)
  pct_of_target= 75.00% → score= 55.00  (Watch→Warning boundary)
  pct_of_target= 62.50% → score= 65.00  (mid Warning)
  pct_of_target= 50.00% → score= 75.00  (Warning→Emergency boundary)
  pct_of_target= 25.00% → score= 87.50  (mid Emergency)
  pct_of_target=  0.00% → score=100.00  (no coverage, floor)

Wheat:
  Target: 4.00 months | Current: 4.00 months
  Gap:    0.00 months | Exposure (disrupted supply fraction): 0.000
   current (t=0): reserve_months=4.00  pct_of_target=100.0%  score=30.0
  forecast (t=1): reserve_months=4.00  pct_of_target=100.0%  score=30.0
  forecast (t=2): reserve_months=4.00  pct_of_target=100.0%  score=30.0
  forecast (t=3): reserve_months=4.00  pct_of_target=100.0%  score=30.0

Corn:
  Target: 3.00 months | 

In [13]:
# ══════════════════════════════════════════════════════════════════
# STEP 5: COMBINE ALL INDICATOR SCORES (updated)
# Now includes domestic_reserves_gap alongside other 6 indicators
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 5: Combining all indicator scores ===")

df_prophet_scores = df_to_score[
    df_to_score["indicator"] != "policy_risk_weighted"
][[
    "ds","indicator","commodity","period_type",
    "yhat","indicator_score","indicator_score_lower","indicator_score_upper"
]].copy()
df_prophet_scores["ds"] = df_prophet_scores["ds"].astype("datetime64[us]")

df_policy_aligned = df_policy_scores[[
    "ds","indicator","commodity","period_type",
    "yhat","indicator_score","indicator_score_lower","indicator_score_upper"
]].copy()
df_policy_aligned["ds"] = df_policy_aligned["ds"].astype("datetime64[us]")

df_reserves_aligned = df_reserves_scores[[
    "ds","indicator","commodity","period_type",
    "yhat","indicator_score","indicator_score_lower","indicator_score_upper"
]].copy()
df_reserves_aligned["ds"] = df_reserves_aligned["ds"].astype("datetime64[us]")

df_all_scores = pd.concat([
    df_prophet_scores,
    df_policy_aligned,
    df_reserves_aligned    # ← NEW
], ignore_index=True)

df_all_scores["ds"] = pd.to_datetime(df_all_scores["ds"])
df_all_scores = df_all_scores.drop_duplicates(
    subset=["ds","indicator","commodity","period_type"],
    keep="last"
).sort_values(["commodity","indicator","ds"]).reset_index(drop=True)

print(f"All indicator scores: {df_all_scores.shape}")
print(f"Indicators: {sorted(df_all_scores['indicator'].unique())}")

save_to_lakehouse(df_all_scores, "prophet_indicator_scores")

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 15, Finished, Available, Finished, False)


=== Step 5: Combining all indicator scores ===
All indicator scores: (160, 8)
Indicators: ['bdi_ratio', 'demand_ratio', 'domestic_reserves_gap', 'fob_price_deviation', 'ksa_deviation', 'policy_risk_weighted', 'ppi_deviation', 'stu_deviation']
✓ srm.prophet_indicator_scores: 160 rows saved


In [14]:
# ══════════════════════════════════════════════════════════════════
# STEP 6: SCORE REVIEW 
# Shows: raw converted value + score side by side
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 6: Score review ===")

all_periods    = pd.to_datetime([CURRENT_MONTH] + list(FORECAST_MONTHS))
period_labels  = [p.strftime('%b %Y') for p in all_periods]

# NEW
REFERENCE_THRESHOLDS = {
    "stu_deviation":         {"Low":-2.5, "Watch":-5.0, "Warning":-7.5, "Emergency":-10.0},   # UPDATED
    "demand_ratio":          {"Low":0, "Watch":10, "Warning":25, "Emergency":">=25"},          # unchanged
    "ppi_deviation":         {"Low":3, "Watch":0, "Warning":-3, "Emergency":-3},                # UPDATED
    "ksa_deviation":         {"Low":15, "Watch":25, "Warning":50, "Emergency":">=50"},          # unchanged
    "policy_risk_weighted":  {"Low":10, "Watch":40, "Warning":70, "Emergency":100},             # unchanged
    "bdi_ratio":             {"Low":30, "Watch":60, "Warning":90, "Emergency":">=90"},          # unchanged
    "domestic_reserves_gap": {"Low":100, "Watch":75, "Warning":50, "Emergency":"<=50"},         # UPDATED
    "fob_price_deviation":   {"Low":5, "Watch":10, "Warning":20, "Emergency":">=20"},           # NEW
}

for commodity in COMMODITIES:
    print(f"\n{'═'*105}")
    print(f"  {commodity}")
    print(f"{'═'*105}")
    period_header = "".join(f"{lbl:>18}" for lbl in period_labels)
    print(f"{'Indicator':<28} {'Wt':>4}  {period_header}")
    print(f"{'-'*105}")

    for indicator in INDICATORS_ORDERED:
        weight = EFFECTIVE_WEIGHTS.get(indicator, 0)
        print(f"{INDICATOR_LABELS[indicator]:<28} {weight*100:>3.0f}%  ", end="")

        for period in all_periods:
            row = df_all_scores[
                (df_all_scores["commodity"]==commodity) &
                (df_all_scores["indicator"]==indicator) &
                (df_all_scores["ds"]==period)
            ]
            if row.empty:
                print(f"{'N/A':>18}", end="")
            else:
                score = row["indicator_score"].values[0]
                raw   = row["yhat"].values[0]
                # Show converted raw value
                if indicator == "stu_deviation":
                    b = baseline_lookup.get((commodity,"stu_ratio"),26.86)
                    disp = (raw / b) * 100
                elif indicator == "ppi_deviation":
                    b = baseline_lookup.get((commodity,"ppi_deviation"),100.0)
                    disp = raw + b - 100
                elif indicator == "bdi_ratio":
                    disp = min(100, max(0,(raw-0.5)/1.5*100))
                elif indicator == "ksa_deviation":
                    b = baseline_lookup.get(
                        (commodity,"ksa_top3_pct"),
                        baseline_lookup.get((commodity,"ksa_deviation"),85.0)
                    )
                    disp = raw + b
                elif indicator == "demand_ratio":
                    disp = raw   # raw here is already prophet_value (the ratio); convert:
                    disp = (raw - 1.0) * 100
                # elif indicator == "fob_price_deviation":              # NEW — add this branch
                #     b = baseline_lookup.get((commodity,"fob_price_deviation"), 300.0)
                #     disp = raw + b
                elif indicator == "fob_price_deviation":
                    b = baseline_lookup.get((commodity,"fob_price_deviation"),300.0)
                    price = np.exp(raw)                    # NEW
                    disp = ((price - b) / b) * 100          # CHANGED
                else:
                    disp = raw
                print(f"{score:>8.1f}({disp:>7.1f})", end="")
        print()

    print(f"\n  Format: score(raw_value)")
    print(f"  Boss thresholds:")
    for indicator in INDICATORS_ORDERED:
        t = REFERENCE_THRESHOLDS[indicator]
        print(f"    {INDICATOR_LABELS[indicator]:<28}: "
              f"Low={t['Low']}  Watch={t['Watch']}  "
              f"Warning={t['Warning']}  Emergency={t['Emergency']}")

print(f"\n=== NOTEBOOK 3 COMPLETE ===")
print(f"""
Tables saved:
  srm.prophet_indicator_scores
  srm.prophet_policy_scores

Next: Notebook 4 — Final Risk Score + Risk Band
""")

StatementMeta(, d6b08c78-bf9d-4d34-854a-3499f51fad6f, 16, Finished, Available, Finished, False)


=== Step 6: Score review ===

═════════════════════════════════════════════════════════════════════════════════════════════════════════
  Wheat
═════════════════════════════════════════════════════════════════════════════════════════════════════════
Indicator                      Wt            Jul 2026          Aug 2026          Sep 2026          Oct 2026
---------------------------------------------------------------------------------------------------------
Global Stock-to-Use Ratio     15%       0.0(   -2.3)     0.0(    0.2)     0.0(    0.7)     0.0(    1.2)
Import Demand Pressure         5%      18.6(  -14.3)    18.6(  -14.3)    18.6(  -14.3)    18.6(  -14.3)
Production Potential Index     5%      75.0(   -0.8)    55.0(    2.4)    55.0(    2.5)    55.0(    2.5)
KSA Import Concentration      15%      93.5(   87.1)    97.3(   94.5)    97.4(   94.8)    97.6(   95.1)
Export Restriction Status     15%       7.5(   17.5)     7.2(   17.2)     6.8(   16.8)     6.5(   16.5)
Logistic Disrup